# Sri Lanka Weather Analytics - Spark Data Loader

This notebook sets up the Spark session and loads weather and location data with proper schema definitions and null handling.

**Requirements:** 4.4, 7.3

## 1. Setup and Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, FloatType, 
    StringType, DateType
)
from pyspark.sql.functions import (
    col, to_date, when, coalesce, lit, avg, sum as spark_sum,
    month, year, dayofmonth, count
)
import os

## 2. Create Spark Session

In [ ]:
# Create Spark session with appropriate configuration
spark = SparkSession.builder \
    .appName("SriLankaWeatherAnalytics") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .config("spark.sql.session.timeZone", "Asia/Colombo") \
    .getOrCreate()

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## 3. Define Schemas

In [ ]:
# Weather data schema
weather_schema = StructType([
    StructField("location_id", IntegerType(), nullable=False),
    StructField("date", StringType(), nullable=False),
    StructField("weather_code", IntegerType(), nullable=True),
    StructField("temperature_2m_max", FloatType(), nullable=True),
    StructField("temperature_2m_min", FloatType(), nullable=True),
    StructField("temperature_2m_mean", FloatType(), nullable=True),
    StructField("apparent_temperature_max", FloatType(), nullable=True),
    StructField("apparent_temperature_min", FloatType(), nullable=True),
    StructField("apparent_temperature_mean", FloatType(), nullable=True),
    StructField("daylight_duration", FloatType(), nullable=True),
    StructField("sunshine_duration", FloatType(), nullable=True),
    StructField("precipitation_sum", FloatType(), nullable=True),
    StructField("rain_sum", FloatType(), nullable=True),
    StructField("precipitation_hours", FloatType(), nullable=True),
    StructField("wind_speed_10m_max", FloatType(), nullable=True),
    StructField("wind_gusts_10m_max", FloatType(), nullable=True),
    StructField("wind_direction_10m_dominant", FloatType(), nullable=True),
    StructField("shortwave_radiation_sum", FloatType(), nullable=True),
    StructField("et0_fao_evapotranspiration", FloatType(), nullable=True),
    StructField("sunrise", StringType(), nullable=True),
    StructField("sunset", StringType(), nullable=True)
])

# Location data schema
location_schema = StructType([
    StructField("location_id", IntegerType(), nullable=False),
    StructField("latitude", FloatType(), nullable=True),
    StructField("longitude", FloatType(), nullable=True),
    StructField("elevation", IntegerType(), nullable=True),
    StructField("utc_offset_seconds", IntegerType(), nullable=True),
    StructField("timezone", StringType(), nullable=True),
    StructField("timezone_abbreviation", StringType(), nullable=True),
    StructField("city_name", StringType(), nullable=False)
])

print("Schemas defined successfully!")

## 4. Load Weather Data

In [ ]:
# Define data paths (adjust based on your environment)
weather_path = "../dataset/weatherData.csv"
location_path = "../dataset/locationData.csv"

# Load weather data with schema
weather_df = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("nullValue", "") \
    .option("nanValue", "NaN") \
    .schema(weather_schema) \
    .csv(weather_path)

print(f"Weather records loaded: {weather_df.count()}")
weather_df.printSchema()

## 5. Parse Dates and Extract Components

In [ ]:
# Parse date from M/D/YYYY format
weather_df = weather_df.withColumn(
    "parsed_date",
    to_date(col("date"), "M/d/yyyy")
)

# Extract date components for easier analysis
weather_df = weather_df \
    .withColumn("year", year(col("parsed_date"))) \
    .withColumn("month", month(col("parsed_date"))) \
    .withColumn("day", dayofmonth(col("parsed_date")))

# Verify date parsing
invalid_dates = weather_df.filter(col("parsed_date").isNull()).count()
print(f"Records with invalid dates: {invalid_dates}")

# Show sample dates
weather_df.select("date", "parsed_date", "year", "month", "day").show(5)

## 6. Handle Null Values

In [ ]:
# Check null counts before handling
print("Null counts per column (before handling):")
for column in weather_df.columns:
    null_count = weather_df.filter(col(column).isNull()).count()
    if null_count > 0:
        print(f"  {column}: {null_count}")

In [ ]:
# Handle null values in aggregation-safe columns
# These columns use 0 as default for null (safe for sum operations)
null_safe_columns = [
    "precipitation_hours",
    "precipitation_sum",
    "rain_sum"
]

for column in null_safe_columns:
    weather_df = weather_df.withColumn(
        column,
        coalesce(col(column), lit(0.0))
    )

print("Null handling applied to precipitation columns.")

## 7. Load Location Data

In [ ]:
# Load location data with schema
location_df = spark.read \
    .option("header", "true") \
    .option("mode", "PERMISSIVE") \
    .option("nullValue", "") \
    .schema(location_schema) \
    .csv(location_path)

print(f"Location records loaded: {location_df.count()}")
location_df.show()

## 8. Validate Data Quality

In [ ]:
# Check for orphan weather records (no matching location)
weather_location_ids = weather_df.select("location_id").distinct()
location_ids = location_df.select("location_id").distinct()

orphan_ids = weather_location_ids.subtract(location_ids)
orphan_count = orphan_ids.count()

print(f"Unique location IDs in weather data: {weather_location_ids.count()}")
print(f"Unique location IDs in location data: {location_ids.count()}")
print(f"Orphan weather records (no matching location): {orphan_count}")

if orphan_count > 0:
    print("\nOrphan location IDs:")
    orphan_ids.show()

## 9. Join Weather and Location Data

In [ ]:
# Join datasets on location_id
joined_df = weather_df.join(
    location_df,
    on="location_id",
    how="inner"
)

print(f"Joined records: {joined_df.count()}")
print(f"\nJoined schema:")
joined_df.printSchema()

In [ ]:
# Show sample joined data
joined_df.select(
    "city_name", "parsed_date", "temperature_2m_mean", 
    "precipitation_hours", "et0_fao_evapotranspiration"
).show(10, truncate=False)

## 10. Null-Safe Aggregation Functions

In [ ]:
def null_safe_avg(df, column, group_by_cols=None):
    """
    Calculate average excluding null values.
    
    Args:
        df: DataFrame
        column: Column to average
        group_by_cols: Optional list of columns to group by
        
    Returns:
        DataFrame with average calculation
    """
    if group_by_cols:
        return df.filter(col(column).isNotNull()) \
            .groupBy(group_by_cols) \
            .agg(avg(col(column)).alias(f"avg_{column}"))
    else:
        return df.filter(col(column).isNotNull()) \
            .agg(avg(col(column)).alias(f"avg_{column}"))


def null_safe_sum(df, column, group_by_cols=None):
    """
    Calculate sum with nulls treated as 0.
    
    Args:
        df: DataFrame
        column: Column to sum
        group_by_cols: Optional list of columns to group by
        
    Returns:
        DataFrame with sum calculation
    """
    df_safe = df.withColumn(column, coalesce(col(column), lit(0.0)))
    
    if group_by_cols:
        return df_safe.groupBy(group_by_cols) \
            .agg(spark_sum(col(column)).alias(f"sum_{column}"))
    else:
        return df_safe.agg(spark_sum(col(column)).alias(f"sum_{column}"))

print("Null-safe aggregation functions defined.")

## 11. Example Aggregations

In [ ]:
# Example: Average temperature by city (null-safe)
print("Average temperature by city:")
avg_temp = null_safe_avg(joined_df, "temperature_2m_mean", ["city_name"])
avg_temp.orderBy("avg_temperature_2m_mean", ascending=False).show(10)

In [ ]:
# Example: Total precipitation hours by district and year
print("Total precipitation hours by district and year:")
precip_by_year = null_safe_sum(joined_df, "precipitation_hours", ["city_name", "year"])
precip_by_year.orderBy("city_name", "year").show(20)

## 12. Data Summary

In [ ]:
# Summary statistics for key columns
print("Summary statistics for key weather metrics:")
joined_df.select(
    "temperature_2m_mean",
    "precipitation_hours",
    "shortwave_radiation_sum",
    "et0_fao_evapotranspiration"
).describe().show()

In [ ]:
# Date range in dataset
date_range = joined_df.agg(
    {"parsed_date": "min", "parsed_date": "max"}
).collect()[0]

min_date = joined_df.agg({"parsed_date": "min"}).collect()[0][0]
max_date = joined_df.agg({"parsed_date": "max"}).collect()[0][0]

print(f"Date range: {min_date} to {max_date}")

## 13. Cache DataFrames for Reuse

In [ ]:
# Cache the joined DataFrame for efficient reuse in subsequent analyses
joined_df.cache()
print(f"Joined DataFrame cached. Record count: {joined_df.count()}")

---

## Data Loading Complete!

The following DataFrames are now available for analysis:
- `weather_df`: Raw weather data with parsed dates
- `location_df`: Location/district information
- `joined_df`: Combined weather and location data (cached)

Utility functions available:
- `null_safe_avg(df, column, group_by_cols)`: Calculate average excluding nulls
- `null_safe_sum(df, column, group_by_cols)`: Calculate sum with nulls as 0